In [ ]:
from planner import solve_foundation

result = solve_foundation(timeout=3.0)
print(f"Status: {result.status}; elapsed: {result.elapsed_seconds:.6f}s")
if result.status == "FEASIBLE":
    for semester in range(1, 9):
        courses = sorted(c for c, s in result.assignment.items() if s == semester)
        print(f"Semester {semester}: {', '.join(courses) or '(empty)'}")
    print("Conditional fixed-course baseline; not a full graduation plan.")
for error in result.validation_errors:
    print(error)


The following code is to fetch data from the nusmods api. 

Warning: It fetches ALL the courses data and takes around 20-30 mins to finish executing. Do not run unless sure you want it all. The `nusmods_modules.json` file already has the necessary courses for the first version of the project

In [5]:
import json
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

BASE_URL = "https://api.nusmods.com/v2"
ACAD_YEAR = "2026-2027"
OUTPUT_FILE = Path("nusmods_modules.json")
TIMEOUT = 30


def get_json(url):
    response = requests.get(url, timeout=TIMEOUT)
    response.raise_for_status()
    return response.json()


def fetch_module(module):
    code = module["moduleCode"]
    url = f"{BASE_URL}/{ACAD_YEAR}/modules/{code}.json"

    try:
        details = get_json(url)

        semester_data = details.get("semesterData", [])

        # Keep one entry per module, with its semesters in a list.
        return {
            "module_code": details.get("moduleCode", code),
            "module_credit": details.get("moduleCredit"),
            "semesters": [
                {
                    "semester_number": sem.get("semester"),
                    "exam_date_time": sem.get("examDate"),
                    "exam_duration": sem.get("examDuration"),
                }
                for sem in semester_data
            ],
            "preclusions": details.get("preclusion"),
            "prereq_tree": details.get("prereqTree"),
        }

    except requests.RequestException as error:
        print(f"Failed to fetch {code}: {error}")
        return None


def main():
    print(f"Fetching module list for AY{ACAD_YEAR}...")

    modules = get_json(
        f"{BASE_URL}/{ACAD_YEAR}/moduleList.json"
    )

    results = []
    failed = []

    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {
            executor.submit(fetch_module, module): module["moduleCode"]
            for module in modules
        }

        for index, future in enumerate(as_completed(futures), 1):
            code = futures[future]

            try:
                result = future.result()
                if result is not None:
                    results.append(result)
                else:
                    failed.append(code)
            except Exception as error:
                print(f"Error processing {code}: {error}")
                failed.append(code)

            if index % 100 == 0:
                print(f"Processed {index}/{len(modules)} modules")

    # Sort by module code for easier browsing.
    results.sort(key=lambda module: module["module_code"])

    output = {
        "academic_year": ACAD_YEAR,
        "total_modules_fetched": len(results),
        "failed_module_codes": failed,
        "modules": results,
    }

    with OUTPUT_FILE.open("w", encoding="utf-8") as file:
        json.dump(
            output,
            file,
            indent=2,
            ensure_ascii=False
        )

    print(f"\nSaved {len(results)} modules to {OUTPUT_FILE.resolve()}")
    print(f"Failed modules: {len(failed)}")


if __name__ == "__main__":
    main()


Fetching module list for AY2026-2027...
Processed 100/6765 modules
Processed 200/6765 modules
Processed 300/6765 modules
Processed 400/6765 modules
Processed 500/6765 modules
Processed 600/6765 modules
Processed 700/6765 modules
Processed 800/6765 modules
Failed to fetch CE5111: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Processed 900/6765 modules
Failed to fetch CE5214QB: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Processed 1000/6765 modules
Processed 1100/6765 modules
Processed 1200/6765 modules
Processed 1300/6765 modules
Processed 1400/6765 modules
Processed 1500/6765 modules
Processed 1600/6765 modules
Processed 1700/6765 modules
Processed 1800/6765 modules
Processed 1900/6765 modules
Processed 2000/6765 modules
Processed 2100/6765 modules
Processed 2200/6765 modules
Processed 2300/6765 modules
P